# Classify all beach cells with best ConvNeXt Tiny

This notebook reads all complete cell matrices from `Data/Raw/beach_all`, loads the best saved context-aware ConvNeXt Tiny model, and writes per-date classification CSV files to `Results/all_cells_classification`.

It does not copy any input `.txt` files from the external drive.

In [1]:
from pathlib import Path
import sys

import torch

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (
    (PROJECT_ROOT / "Source").exists()
    or (PROJECT_ROOT / "Data").exists()
    or (PROJECT_ROOT / "pmaciag-volume-beach").exists()
):
    PROJECT_ROOT = PROJECT_ROOT.parent
if (PROJECT_ROOT / "pmaciag-volume-beach").exists():
    PROJECT_ROOT = PROJECT_ROOT / "pmaciag-volume-beach"
if not ((PROJECT_ROOT / "Source").exists() or (PROJECT_ROOT / "Data").exists()):
    raise RuntimeError("Could not locate pmaciag-volume-beach project directory.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Source.inference import (
    classify_date_directories,
    discover_date_directories,
    load_inference_metadata,
)
from Source.training import get_device

In [2]:
SOURCE_ROOT = PROJECT_ROOT / "Data" / "Raw" / "beach_all"
MODEL_RESULTS_DIR = PROJECT_ROOT / "Results" / "convnext_tiny"
OUTPUT_ROOT = PROJECT_ROOT / "Results" / "all_cells_classification"

CHECKPOINT_PATH = MODEL_RESULTS_DIR / "convnext_tiny_best.pt"
INFERENCE_METADATA_PATH = MODEL_RESULTS_DIR / "inference_metadata.json"

BATCH_SIZE = 128
NUM_WORKERS = 0
BASE_SUFFIX = "r"

print(f"Project root:       {PROJECT_ROOT}", flush=True)
print(f"Source root:        {SOURCE_ROOT}", flush=True)
print(f"Model checkpoint:   {CHECKPOINT_PATH}", flush=True)
print(f"Inference metadata: {INFERENCE_METADATA_PATH}", flush=True)
print(f"Output root:        {OUTPUT_ROOT}", flush=True)

Project root:       /workspace/pmaciag-volume-beach


Source root:        /workspace/pmaciag-volume-beach/Data/Raw/beach_all


Model checkpoint:   /workspace/pmaciag-volume-beach/Results/convnext_tiny/convnext_tiny_best.pt


Inference metadata: /workspace/pmaciag-volume-beach/Results/convnext_tiny/inference_metadata.json


Output root:        /workspace/pmaciag-volume-beach/Results/all_cells_classification


In [3]:
print("Checking required input paths...", flush=True)
if not SOURCE_ROOT.exists():
    raise FileNotFoundError(f"External source directory not found: {SOURCE_ROOT}")

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"ConvNeXt checkpoint not found: {CHECKPOINT_PATH}")

if not INFERENCE_METADATA_PATH.exists():
    raise FileNotFoundError(
        f"Inference metadata not found: {INFERENCE_METADATA_PATH}. "
        "Rerun the ConvNeXt training notebook so it saves inference_metadata.json."
    )

print("Loading inference metadata...", flush=True)
metadata = load_inference_metadata(INFERENCE_METADATA_PATH)
idx_to_class_code = {
    int(index): str(class_code)
    for index, class_code in metadata["idx_to_class_code"].items()
}
if "context_radius" in metadata:
    context_radius = int(metadata["context_radius"])
elif "context_radius" in metadata.get("best_config", {}):
    context_radius = int(metadata["best_config"]["context_radius"])
else:
    legacy_context_size = int(metadata.get("context_size", 0))
    context_radius = 0 if legacy_context_size <= 0 else legacy_context_size // 2

print(f"Image size:       {metadata['image_size']}", flush=True)
print(f"Context radius:   {context_radius}", flush=True)
print(f"Context window:   {2 * context_radius + 1}x{2 * context_radius + 1}", flush=True)
print(f"Feature suffixes: {metadata['feature_suffixes']}", flush=True)
print(f"Class mapping:    {idx_to_class_code}", flush=True)

Checking required input paths...


Loading inference metadata...


Image size:       64


Context radius:   1


Context window:   3x3


Feature suffixes: ['r', 'g', 'b', 's', 'd']


Class mapping:    {0: '0', 1: '1', 2: '2', 3: '3', 4: '4', 5: '5', 6: '6', 7: '7', 8: '8', 9: '9'}


In [4]:
print("Selecting compute device...", flush=True)
device = get_device()
print(f"Device: {device}", flush=True)
if device.type == "cuda":
    print(f"CUDA devices available: {torch.cuda.device_count()}", flush=True)
    for index in range(torch.cuda.device_count()):
        print(f"CUDA device {index}: {torch.cuda.get_device_name(index)}", flush=True)

Selecting compute device...


Device: cuda


CUDA devices available: 1


CUDA device 0: NVIDIA RTX 6000 Ada Generation


In [5]:
print("Discovering date folders...", flush=True)
date_dirs = discover_date_directories(SOURCE_ROOT)
print(f"Date folders found: {len(date_dirs)}", flush=True)
print("This notebook uses a single-pass workflow: each date folder is scanned once using BASE_SUFFIX, then classified immediately.", flush=True)

Discovering date folders...


Date folders found: 15


This notebook uses a single-pass workflow: each date folder is scanned once using BASE_SUFFIX, then classified immediately.


In [6]:
print("Starting all-cells classification...", flush=True)
print("All-cells classification finished.", flush=True)
written_summary = classify_date_directories(
    date_dirs=date_dirs,
    source_root=SOURCE_ROOT,
    output_root=OUTPUT_ROOT,
    checkpoint_path=CHECKPOINT_PATH,
    metadata=metadata,
    idx_to_class_code=idx_to_class_code,
    context_radius=context_radius,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    base_suffix=BASE_SUFFIX,
    device=device,
)
print("All-cells classification finished.", flush=True)
written_summary

Starting all-cells classification...


All-cells classification finished.


Classifying date folders:   0%|          | 0/15 [00:00<?, ?it/s]

Classifying 20180722 (46,483 cells):   0%|          | 0/364 [00:00<?, ?it/s]

Finished 20180722 on cuda: classified 46,483, skipped 0, saved /workspace/pmaciag-volume-beach/Results/all_cells_classification/20180722/classification.csv


...

Classifying 20240623 (72,043 cells):   0%|          | 0/563 [00:00<?, ?it/s]

Finished 20240623 on cuda: classified 72,043, skipped 0, saved /workspace/pmaciag-volume-beach/Results/all_cells_classification/20240623/classification.csv


Classifying 20240806 (58,149 cells):   0%|          | 0/455 [00:00<?, ?it/s]

Finished 20240806 on cuda: classified 58,149, skipped 0, saved /workspace/pmaciag-volume-beach/Results/all_cells_classification/20240806/classification.csv
All-cells classification finished.


,date,classified_cells,incomplete_candidates,device,output_path
0,20180722,46483,0,cuda,/workspace/pmaciag-volume-beach/Results/all_ce...
1,20180731,54304,0,cuda,/workspace/pmaciag-volume-beach/Results/all_ce...
2,20180812,67022,0,cuda,/workspace/pmaciag-volume-beach/Results/all_ce...
3,20180826,62815,0,cuda,/workspace/pmaciag-volume-beach/Results/all_ce...
4,20180910,84196,0,cuda,/workspace/pmaciag-volume-beach/Results/all_ce...
5,20180915,52432,0,cuda,/workspace/pmaciag-volume-beach/Results/all_ce...
6,20190822,61357,0,cuda,/workspace/pmaciag-volume-beach/Results/all_ce...
7,20220826,74975,0,cuda,/workspace/pmaciag-volume-beach/Results/all_ce...
8,20220910,50070,0,cuda,/workspace/pmaciag-volume-beach/Results/all_ce...
9,20220924,50765,0,cuda,/workspace/pmaciag-volume-beach/Results/all_ce...
